## Learning notes — 02. Process Customers Data

### Learning objectives
Follow **02. Process Customers Data** from its source inputs through its operations to the archived results.
- Explain stream, autoloader, ctas, scd, profiling as used in this lesson.

### Source-specific learning focus
- Follow incremental ingestion from discovery to persisted results.
- Separate file discovery, parsing and schema tracking.
- Distinguish query results from materialized table state.
- Trace sequenced changes into current-state or history-preserving records.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.


## Learning notes — supplied input snapshot

Path: `data/circuitbox/customers/customers_day1.json`

SHA-256: `877701de9fc03ce84fb0c14c7a64a48444595a9251b9f2bd4414a6b6a0ad0512`

Complete supplied text (not generated output):

```json
{"customer_id": 1001, "customer_name": "Richard Cox", "date_of_birth": "1996-10-25", "telephone": "+1 6680703335", "email": "richard.cox@gmail.com", "created_date": "2024-10-01"}
{"customer_id": 1002, "customer_name": "Carla Morton", "date_of_birth": "2004-06-21", "telephone": "+1 8616454195", "email": null, "created_date": "2024-10-01"}
{"customer_id": 1003, "customer_name": "Billy Scott", "date_of_birth": "1997-03-17", "telephone": "+1 5544387564", "email": "billy.scott@hotmail.com", "created_date": "2024-10-01"}
{"customer_id": 1004, "customer_name": "Lori Mason", "date_of_birth": "2002-11-01", "telephone": null, "email": "lori.mason@yahoo.com", "created_date": "2024-10-01"}
{"customer_id": 1005, "customer_name": "Jennifer Haas", "date_of_birth": "2001-04-03", "telephone": "+1 4725460000", "email": "jennifer.haas@gmail.com", "created_date": "2024-10-01"}
{"customer_id": 1006, "customer_name": "Joseph Keller", "date_of_birth": "2003-02-11", "telephone": "+1 3817867756", "email": null, "created_date": "2024-10-01"}
{"customer_id": 1007, "customer_name": "Jessica Harris", "date_of_birth": "2004-04-19", "telephone": "+1 8604009935", "email": null, "created_date": "2024-10-01"}
{"customer_id": 1008, "customer_name": null, "date_of_birth": "2003-09-05", "telephone": "+1 1448753611", "email": "william.carter@hotmail.com", "created_date": "2024-10-01"}
{"customer_id": 1009, "customer_name": "Shannon Austin", "date_of_birth": "2002-03-22", "telephone": "+1 4594705629", "email": "austin.shannon@outlook.com", "created_date": "2024-10-01"}
{"customer_id": 1010, "customer_name": "Andrew Phillips", "date_of_birth": "2003-04-17", "telephone": "+1 4079273853", "email": "andrew.phillips@yahoo.com", "created_date": "2024-10-01"}
```

## Learning notes — supplied input snapshot

Path: `data/circuitbox/customers/customers_day2.json`

SHA-256: `b24ecb34aaee37bda161ef5798360dbc94b6abbfc8413d91c7f0c0d73d4d4d39`

Complete supplied text (not generated output):

```json
{"customer_id": 1002, "customer_name": "Carla Morton", "date_of_birth": "2004-06-21", "telephone": "+1 8616454195", "email": "carla.morton@yahoo.com", "created_date": "2025-01-14"}
{"customer_id": 1004, "customer_name": "Lori Mason", "date_of_birth": "2002-11-01", "telephone": "+1 0498301620", "email": "lori.mason@yahoo.com", "created_date": "2025-01-14"}
{"customer_id": 1008, "customer_name": "William Carter", "date_of_birth": "2003-09-05", "telephone": "+1 1448753611", "email": "william.carter@hotmail.com", "created_date": "2025-01-14"}
{"customer_id": 1009, "customer_name": "Shannon Austin", "date_of_birth": "2002-03-22", "telephone": "+1 4594705629", "email": "shannon.austin@yahoo.com", "created_date": "2025-01-14"}
{"customer_id": 1011, "customer_name": "Martin Thomas DDS", "date_of_birth": "2002-05-01", "telephone": "+1 0201457151", "email": "martin.dds@gmail.com", "created_date": "2025-01-14"}
{"customer_id": 1012, "customer_name": "Rachel Thomas", "date_of_birth": "2003-02-17", "telephone": "+1 5437271621", "email": "rachel.thomas@gmail.com", "created_date": "2025-01-14"}
{"customer_id": 1013, "customer_name": "Dr. Daniel Hall", "date_of_birth": "2004-03-21", "telephone": "+1 7544137170", "email": "dr.hall@hotmail.com", "created_date": "2025-01-14"}
```

# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Process the Customers Data 
1. Ingest the data into the data lakehouse - bronze_customers
1. Perform data quality checks and transform the data as required - silver_customers_clean
1. Apply changes to the Customers data - silver_customers

![dea06-02-process-customer-data-1.png](images/dea06-02-process-customer-data-1.png)

### 1. Ingest the data into the data lakehouse - bronze_customers

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**autoloader**: Auto Loader discovers arriving files. File format/options and schema tracking determine parsing; discovery is distinct from downstream transformations.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REFRESH STREAMING TABLE bronze_customers
COMMENT 'Raw customers data ingested from the source system operational data'
TBLPROPERTIES ('quality' = 'bronze')
AS 
SELECT *,
       _metadata.file_path AS input_file_path,
       CURRENT_TIMESTAMP AS ingestion_timestamp
FROM cloud_files(
  '/Volumes/circuitbox/landing/operational_data/customers/',
  'json',
  map('cloudFiles.inferColumnTypes', 'true')
);
```

In [ ]:
# Archived output only; no computation.

Name,Type
created_date,string
customer_id,bigint
customer_name,string
date_of_birth,string
email,string
telephone,string
_rescued_data,string
input_file_path,string
ingestion_timestamp,timestamp


### 2. Perform data quality checks and transform the data as required - silver_customers_clean

![dea06-02-process-customer-data-2](images/dea06-02-process-customer-data-2.png)

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REFRESH STREAMING TABLE silver_customers_clean(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_customer_name EXPECT (customer_name IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_telephone EXPECT (LENGTH(telephone) >= 10),
  CONSTRAINT valid_email EXPECT (email IS NOT NULL),
  CONSTRAINT valid_date_of_birth EXPECT(date_of_birth >= '1920-01-01') 
)
COMMENT 'Cleaned customers data'
TBLPROPERTIES ('quality' = 'silver')
AS
SELECT customer_id,
       customer_name,
       CAST(date_of_birth AS DATE) AS date_of_birth,
       telephone,
       email,
       CAST(created_date AS DATE) AS created_date
  FROM STREAM(LIVE.bronze_customers)
```

### 3. Apply changes to the Customers data - silver_customers

![dea06-02-process-customer-data-3](images/dea06-02-process-customer-data-3.png)

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**scd**: Change application uses keys and sequencing to resolve events. SCD type 1 replaces current values; type 2 retains validity intervals. Verify the actual configured type and event ordering.

**Unexecuted teaching example (illustrative, not an archived or generated result):** For a customer address change, SCD1 replaces the address; SCD2 can keep the old and new address in separate validity intervals.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REFRESH STREAMING TABLE silver_customers
COMMENT 'SCD Type 1 customers data'
TBLPROPERTIES ('quality' = 'silver');
```

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**scd**: Change application uses keys and sequencing to resolve events. SCD type 1 replaces current values; type 2 retains validity intervals. Verify the actual configured type and event ordering.

**Unexecuted teaching example (illustrative, not an archived or generated result):** For a customer address change, SCD1 replaces the address; SCD2 can keep the old and new address in separate validity intervals.

Archived Databricks source — shown for reading, not Python execution.

```sql
APPLY CHANGES INTO LIVE.silver_customers
FROM STREAM(LIVE.silver_customers_clean)
KEYS (customer_id)
SEQUENCE BY created_date
STORED AS SCD TYPE 1; -- Optional. Type 1 is the default value
```